In [1]:
import time

import numpy as np
import torch

import gpplus
from data.data_gen import load_data_buckling_MF
from gpplus.models import GPR
from gpplus.training.eval import evaluate_gp_model
from gpplus.utils import set_seed
from gpplus.utils.latent_reps import get_latent_representations, plot_encoders

from sklearn.metrics import mean_squared_error

In [2]:
def compute_metrics(y_true, y_hat, output_std=None, start_time=None):
    """
    Compute basic metrics for predictions.

    Args:
        y_true: True values (1D array)
        y_hat: Predicted values (1D array)
        output_std: Standard deviation of predictions (optional)
        start_time: Start time for timing (optional)

    Returns:
        dict: Dictionary with computed metrics
    """
    # Convert to numpy if needed
    if isinstance(y_true, torch.Tensor):
        y_true = y_true.detach().cpu().numpy().reshape(-1)
    if isinstance(y_hat, torch.Tensor):
        y_hat = y_hat.detach().cpu().numpy().reshape(-1)
    if output_std is not None and isinstance(output_std, torch.Tensor):
        output_std = output_std.detach().cpu().numpy().reshape(-1)

    # Only add time if start_time is provided
    if start_time is not None:
        metrics = {
            "Time": time.time() - start_time,
            "RRMSE": np.sqrt(mean_squared_error(y_true, y_hat)) / y_true.std(),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_hat)),
            "MSE": mean_squared_error(y_true, y_hat),
            # "MAE": mean_absolute_error(y_true, y_hat),
            # "R2": r2_score(y_true, y_hat)
        }
    else:
        metrics = {
            "RRMSE": np.sqrt(mean_squared_error(y_true, y_hat)) / y_true.std(),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_hat)),
            "MSE": mean_squared_error(y_true, y_hat),
            # "MAE": mean_absolute_error(y_true, y_hat),
            # "R2": r2_score(y_true, y_hat)
        }

    # Add NIS if output_std is provided
    if output_std is not None:
        z = 1.96
        L = y_hat - z * output_std
        U = y_hat + z * output_std
        width = U - L
        below = (L - y_true) * (y_true < L)
        above = (y_true - U) * (y_true > U)
        interval_score = width + (2 / 0.05) * below + (2 / 0.05) * above
        NIS = interval_score.mean() / y_true.std()
        metrics["NIS"] = NIS
        return metrics

    return metrics

In [3]:
seed = 42
set_seed(seed)

# Generate training and test data for all fidelity levels s0-s3
sources = ["s0", "s1"]
num_train_per_source = {"s0": 250, "s1": 250}
num_test_per_source = {"s0": 5000, "s1": 5000}  # 10k total / 4 sources = 2.5k per source

# Generate test data
print("\nGenerating test data...")
X_test_data = []
y_test_data = []

data = load_data_buckling_MF(n_train=num_train_per_source, 
    n_test=num_test_per_source, 
    noise_levels=[0.0, 0.0], 
    return_one_hot=True, 
    shuffle=True,
    seed=seed
    )

# Concatenate all test data (10,000x14)
X_test = data["x_test_full"]
y_test = data["y_test_full"]
# Concatenate all training data (400x14)
X_train = data["x_train_full"]
y_train = data["y_train_full"]

cont_cols = [11]
source_cols = [0, 1]

print(f"\nFinal training dataset shape: X={X_train.shape}, y={y_train.shape}")
print(f"Final test dataset shape: X={X_test.shape}, y={y_test.shape}")


# Standardize the data
scalerX = gpplus.utils.StandardScaler()
scalerX.fit(X_train[:,cont_cols])
X_train[:,cont_cols] = scalerX.transform(X_train[:,cont_cols])
X_test[:,cont_cols] = scalerX.transform(X_test[:,cont_cols])


Generating test data...

Final training dataset shape: X=torch.Size([500, 12]), y=torch.Size([500])
Final test dataset shape: X=torch.Size([10000, 12]), y=torch.Size([10000])


In [ ]:
cat_cols = [2, 3, 4, 5, 6, 7, 8, 9, 10] # Uses a single list for grouped OH encodings
t1 = time.time()

source_encoder = gpplus.utils.MatrixEncoder(input_dim=2, 
                initialization="normal", 
                init_std=0.1,
                z_dim=2)

source_encoder2 = gpplus.utils.NeuralEncoder(input_dim=2, 
                architecture_config={"hidden_dims": [], 
                        "activation": "relu", 
                        "dropout": 0.0}, 
                z_dim=2) 

# Create model
kernel = gpplus.kernels.CombinedKernel_MVMF(
    cont_cols=cont_cols,
    cat_cols=cat_cols,
    source_cols=source_cols,
    cat_encoder='matrix'
    # source_encoder=source_encoder,
    # source_encoder=source_encoder2,
)

model_grouped = GPR(
    X_train,
    y_train,
    kernel_module=kernel,
    mean_module=gpplus.means.MultipleMean(encoded_cols=source_cols),
    likelihood=gpplus.likelihoods.MultiLikelihood(encoded_cols=source_cols, training_data=X_train),
    # likelihood=gpytorch.likelihoods.GaussianLikelihood(),
)

num_epochs = 10000
num_runs = 16
lr = 0.1
print(model_grouped)

# Create trainer
trainer = gpplus.training.GPTrainer(
    model=model_grouped,
    num_epochs=num_epochs,
    seed=seed,
    num_runs=num_runs,
    optimizer_kwargs={"lr": lr},
    convergence_patience=50,
    optimizer_class=torch.optim.Adam,
    device="cpu",
)

print("Training model...")
results = trainer.train()

# Evaluate on standardized test data
y_pred_scaled, pred_lower_scaled, pred_upper_scaled, output_std_scaled = evaluate_gp_model(model, X_test)

# Transform predictions back to original scale for proper metrics
y_pred_orig = scalerY.inverse_transform(y_pred_scaled)
output_std_orig = output_std_scaled * scalerY.std  # Scale the uncertainty

metric = compute_metrics(
y_test, y_pred_orig, output_std_orig, start_time=t1
)

print(f"Metrics:")
for k, v in metric.items():
    print(f"  {k}: {v:.4f}")

GPR(
  (likelihood): MultiLikelihood(
    (noise_covar): MultiNoise(
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): MultipleMean(
    (mean0): ZeroMean()
    (mean1): ConstantMean(
      (mean_prior): NormalPrior()
    )
  )
  (covar_module): CombinedKernel_MVMF(
    (cont_kernel): GaussianKernel()
    (cat_kernel): GaussianKernel()
    (source_kernel): GaussianKernel()
    (cat_encoder): MatrixEncoder()
    (source_encoder): OneHotToLatent(
      (fci): Sequential(
        (0): Linear(in_features=2, out_features=2, bias=True)
      )
    )
    (outputscale_constraint): GreaterThan(1.000E-06)
  )
)
Training model...


In [ ]:
from gpplus.utils.latent_reps import get_latent_representations, plot_encoders
  
encoder_data_dict = get_latent_representations(model_grouped, qual_dict={0: 2, 1: 4, 2: 3})
print(encoder_data_dict)

In [ ]:
plot_encoders(model_grouped, qual_dict={0: 2, 1: 4, 2: 3})

In [ ]:
cat_cols = [[2, 3], [4, 5, 6, 7], [8, 9, 10]] # Uses lists within lists for separate OH encodings
t1 = time.time()

source_encoder = gpplus.utils.MatrixEncoder(input_dim=2, 
                initialization="normal", 
                init_std=0.1,
                z_dim=2)

source_encoder2 = gpplus.utils.NeuralEncoder(input_dim=2,
                architecture_config={"hidden_dims": [], 
                        "activation": "relu", 
                        "dropout": 0.0}, 
                z_dim=2)

# Create model
kernel = gpplus.kernels.CombinedKernel_MVMF(
    cont_cols=cont_cols,
    cat_cols=cat_cols,
    source_cols=source_cols,
    cat_encoder='matrix'
    # source_encoder=source_encoder,
    # source_encoder=source_encoder2,
)

model_separate = GPR(
    X_train,
    y_train,
    kernel_module=kernel,
    mean_module=gpplus.means.MultipleMean(encoded_cols=source_cols),
    likelihood=gpplus.likelihoods.MultiLikelihood(encoded_cols=source_cols, training_data=X_train),
    # likelihood=gpytorch.likelihoods.GaussianLikelihood(),

)

num_epochs = 10000
num_runs = 16
lr = 0.1
print(model_separate)

# Create trainer
trainer = gpplus.training.GPTrainer(
    model=model_separate,
    num_epochs=num_epochs,
    seed=seed,
    num_runs=num_runs,
    optimizer_kwargs={"lr": lr},
    convergence_patience=50,
    optimizer_class=torch.optim.Adam,
    device="cpu",
)

print("Training model...")
results = trainer.train()

# Evaluate on standardized test data
y_pred_scaled, pred_lower_scaled, pred_upper_scaled, output_std_scaled = evaluate_gp_model(model_separate, X_test)

# Transform predictions back to original scale for proper metrics
y_pred_orig = scalerY.inverse_transform(y_pred_scaled)
output_std_orig = output_std_scaled * scalerY.std  # Scale the uncertainty

# Compute metrics on original scale
metric = compute_metrics(
y_test, y_pred_orig, output_std_orig, start_time=t1
)

print(f"Metrics:")
for k, v in metric.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
encoder_data_dict = get_latent_representations(model_separate)
print(encoder_data_dict)

In [ ]:
plot_encoders(model_separate)